[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%203/demo_multimodal_autogluon.ipynb)

# ISBA 2411 - Live Demo: Add the columns, watch the score move

**Multimodal classification with AutoGluon.** Text only, then text plus structured columns.

Instructor-run, ~15 min. A GPU runtime speeds up AutoGluon's text model. Pre-run once before class.

We build a small, self-contained product-review dataset (no download), so the demo always runs. Each row has a review, a price, a category, a verified-purchase flag, and a helpful-vote count. The label is whether the product was highly rated.

In [ ]:
# Build the dataset (no download). The text channel is deliberately noisy,
# so text alone is good but not great, and the columns add real signal.
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42); N = 2600
label = rng.integers(0, 2, N)
pos = ["works great", "love it", "exceeded expectations", "sturdy and reliable", "would buy again",
       "excellent quality", "holds up well", "better than described", "fast shipping and solid build"]
neg = ["broke within a week", "poor quality", "waste of money", "stopped working", "very disappointed",
       "cheaply made", "not as described", "fell apart quickly", "would not recommend"]

def make_text(y):
    pool = pos if y == 1 else neg
    if rng.random() < 0.22:                      # noise in the text channel
        pool = neg if pool is pos else pos
    k = int(rng.integers(2, 4))
    return ", ".join(rng.choice(pool, size=k, replace=False)) + "."

text = [make_text(y) for y in label]
helpful = rng.poisson(np.where(label == 1, 6, 3))
verified = (rng.random(N) < np.where(label == 1, 0.71, 0.56))
price = np.where(label == 1, rng.normal(46, 18, N), rng.normal(53, 18, N)).round(2)
cats = ["electronics", "home", "toys", "books", "beauty"]
cpos = np.array([0.16, 0.20, 0.20, 0.24, 0.20]); cpos = cpos / cpos.sum()
cneg = np.array([0.29, 0.22, 0.20, 0.15, 0.14]); cneg = cneg / cneg.sum()
category = [cats[rng.choice(5, p=(cpos if y == 1 else cneg))] for y in label]

df = pd.DataFrame({"text": text, "price": price, "category": category,
                   "verified_purchase": verified, "helpful_votes": helpful, "label": label})
train_df, test_df = train_test_split(df, test_size=0.25, random_state=0, stratify=df["label"])
train_df = train_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
print("train:", len(train_df), " test:", len(test_df))
df.head()

## 1. Text only  (~7 min)

Train AutoGluon on the review text alone. The first run downloads a text backbone.

In [ ]:
!pip -q install autogluon.multimodal
from autogluon.multimodal import MultiModalPredictor
import time

def acc_of(predictor, cols):
    r = predictor.evaluate(test_df[cols], metrics=["accuracy"])
    return r["accuracy"] if isinstance(r, dict) else float(r)

t0 = time.time()
p_text = MultiModalPredictor(label="label", eval_metric="accuracy").fit(
    train_df[["text", "label"]], time_limit=180)
acc_text = acc_of(p_text, ["text", "label"])
print(f"TEXT ONLY accuracy: {acc_text:.3f}   (trained in {time.time()-t0:.0f}s)")

## 2. Text + tabular  (~7 min)

Hand AutoGluon the whole table and let it fuse the modalities.

In [ ]:
p_all = MultiModalPredictor(label="label", eval_metric="accuracy").fit(train_df, time_limit=180)
acc_all = acc_of(p_all, list(test_df.columns))
print(f"TEXT + TABULAR accuracy: {acc_all:.3f}")
print(f"fusion gain over text alone: +{acc_all - acc_text:.3f}")

## 3. The honesty beat: tabular only

Before we celebrate fusion, check whether the columns alone already got there. If they did, the text (and the multimodal cost) added little.

In [ ]:
tab_cols = ["price", "category", "verified_purchase", "helpful_votes", "label"]
p_tab = MultiModalPredictor(label="label", eval_metric="accuracy").fit(train_df[tab_cols], time_limit=120)
acc_tab = acc_of(p_tab, tab_cols)
print(f"TABULAR ONLY accuracy: {acc_tab:.3f}")
print()
print(f"{'text only':<16}{acc_text:.3f}")
print(f"{'tabular only':<16}{acc_tab:.3f}")
print(f"{'text + tabular':<16}{acc_all:.3f}")

## CPU fallback (no AutoGluon, no GPU): the same comparison in seconds

If the AutoGluon install or the GPU is a problem in the room, run this instead. It computes the same three numbers with scikit-learn and reliably shows the same story. This is the insurance path.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

tab = [("cat", OneHotEncoder(handle_unknown="ignore"), ["category"]),
       ("num", StandardScaler(), ["price", "helpful_votes"]),
       ("flag", "passthrough", ["verified_purchase"])]

def run(use_text, use_tab):
    parts = ([("txt", TfidfVectorizer(), "text")] if use_text else []) + (tab if use_tab else [])
    pipe = Pipeline([("ct", ColumnTransformer(parts)), ("lr", LogisticRegression(max_iter=1000))])
    pipe.fit(train_df, train_df["label"])
    return accuracy_score(test_df["label"], pipe.predict(test_df))

print(f"{'text only':<16}{run(True,  False):.3f}")
print(f"{'tabular only':<16}{run(False, True ):.3f}")
print(f"{'text + tabular':<16}{run(True,  True ):.3f}")

## Talking points

- **Fusion wins.** Text plus tabular beats text alone by a clear margin. The structured columns carry signal the words never contained.
- **The honesty beat.** The columns alone were already close. So the real question is not "does multimodal score higher," it is "is the extra modality and its cost worth the last few points for this job?"
- **When multimodal helps.** Fusion pays off when text and structured data carry partly independent signal. When one modality already dominates, adding the other mostly adds cost.
- **AutoML did the choosing.** Nobody picked an architecture. That is the power, and the reason tonight's discussion asks what is left for the data scientist.

*ISBA 2411 - Natural Language Processing & AI - Summer 2026*